# Calculate NDWI from STAC Imagery

This notebook computes the **Normalized Difference Water Index (NDWI)** from optical imagery served via STAC. It supports two collection-specific paths:

- **`sentinel2_ard`** — read Green/NIR from a multi-band `cog` asset (B03/B08).
- **`sentinel-2-c1-l2a`** — read Green/NIR from separate `green` and `nir` assets.

Public catalogs can be used without a token; authenticated data require a token in `.env`.

---

## What is NDWI?

NDWI uses the difference between green and near-infrared (NIR) reflectance to detect water and surface moisture. Values range from **−1** to **+1**:

| Range   | Interpretation        |
|--------|------------------------|
| ≈ 1    | Water bodies (open water, lakes, rivers) |
| ≈ 0    | Bare soil or sparse vegetation |
| ≈ −1   | Dense vegetation or dry surfaces |

## Parameters

- **STAC Item & Collection** — Set in the *Set variables* cell. Use the placeholders `{{STAC_ITEM_LINK}}` and `{{STAC_COLLECTION_NAME}}` when running from a template.
- **Token** — Optional. Required only for authenticated catalogs/COGs; store as `token=...` in a `.env` file in this directory.
- **AOI** — Optional GeoJSON geometry. If empty, a 1000×1000 pixel window from the centre of the image is used.

## Workflow

1. Load the STAC item (using token if provided).
2. Resolve green and NIR bands from the selected collection path.
3. Read band data and apply scale/offset when using separate assets.
4. Compute NDWI: **(Green − NIR) / (Green + NIR)**.
5. Visualise the results.

## Import Required Libraries

In [ ]:
%pip install python-dotenv

In [ ]:
import os
import pystac
import rasterio
from rasterio.windows import Window
from rasterio.mask import mask
from rasterio.warp import transform_geom
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import json
import warnings
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

## Authorisation

For **non-public** datasets, provide a token in a `.env` file in this directory:

```bash
token=your_token_here
```

See [EODH documentation on sensitive data](https://eodatahub.org.uk/docs/documentation/notebooks/sensitive-data/) for details. Public STAC catalogues do not require a token.

In [ ]:
token = os.getenv("token")

# Auth mode for this run: "auto" | "always" | "never"
# - auto: use token only for known authenticated hosts (e.g. eodatahub/dap.ceda)
# - always: force token header for all asset requests
# - never: never send token header
use_auth = "never"


def configure_gdal_auth_for_href(href, token=token, mode=use_auth):
    """Set/clear GDAL auth header for a specific asset URL."""
    if not token or mode == "never":
        os.environ.pop("GDAL_HTTP_HEADERS", None)
        return

    if mode == "always":
        os.environ["GDAL_HTTP_HEADERS"] = f"Authorization: Bearer {token}"
        return

    # Auto mode: only send token to known authenticated endpoints.
    href_lower = str(href).lower()
    auth_hosts = ("eodatahub.org.uk", "dap.ceda.ac.uk")
    if any(host in href_lower for host in auth_hosts):
        os.environ["GDAL_HTTP_HEADERS"] = f"Authorization: Bearer {token}"
    else:
        os.environ.pop("GDAL_HTTP_HEADERS", None)


# Default to no GDAL auth header until an asset URL is known.
os.environ.pop("GDAL_HTTP_HEADERS", None)

## Set variables

Set the STAC item URL and collection name for your scene. When launched from a template, the placeholders below may already be filled.

In [ ]:
stac_item_url = "{{STAC_ITEM_LINK}}"
stac_collection_name = "{{STAC_COLLECTION_NAME}}"
aoi_param = """{{AOI}}""".strip()

## Load STAC item

In [ ]:
try:
    use_stac_auth = token and use_auth in ("auto", "always")
    if use_stac_auth:
        stac_io = pystac.StacIO.default()
        stac_io.headers = {"Authorization": f"Bearer {token}"}
        item = pystac.Item.from_file(stac_item_url, stac_io=stac_io)
    else:
        item = pystac.Item.from_file(stac_item_url)
    print(f"Successfully loaded STAC item: {item.id}")
    print(f"Collection: {stac_collection_name}")
    print(f"Date: {item.datetime}")
except Exception as e:
    print(f"Error loading STAC item: {e}")
    raise

## Determine Area of Interest (AOI)

Define the region to process. If you provide a **GeoJSON** geometry (e.g. Polygon or FeatureCollection), the raster is clipped to it. Otherwise, a **1000×1000 pixel** window is taken from the centre of the image.

In [ ]:
DEFAULT_WINDOW_SIZE = 1000

aoi_geometry = None
clip_window = None
use_windowed_read = False

if aoi_param and aoi_param.strip().lower() not in ("", "none", "null"):
    try:
        # Parse as JSON
        aoi_data = json.loads(aoi_param)

        # Extract geometry from GeoJSON structure
        if aoi_data.get("type") == "FeatureCollection":
            # Extract first geometry from FeatureCollection
            if aoi_data.get("features") and len(aoi_data["features"]) > 0:
                aoi_geometry = aoi_data["features"][0].get("geometry")
        elif aoi_data.get("type") == "Feature":
            # Extract geometry from Feature
            aoi_geometry = aoi_data.get("geometry")
        elif aoi_data.get("type") in ["Polygon", "MultiPolygon", "Point", "LineString"]:
            # Direct geometry object
            aoi_geometry = aoi_data
        else:
            raise ValueError(f"Unsupported GeoJSON type: {aoi_data.get('type')}")

        # Validate geometry was extracted
        if aoi_geometry and aoi_geometry.get("type"):
            print(f"AOI provided: {aoi_geometry['type']} geometry")
            print("Will clip raster to AOI geometry")
        else:
            raise ValueError("Could not extract geometry from GeoJSON")

    except (json.JSONDecodeError, ValueError, KeyError) as e:
        print(f"Warning: Could not parse AOI: {e}. Using default window.")
        aoi_geometry = None
else:
    print("No AOI provided, using default 1000×1000 pixel window")
if aoi_geometry is None:
    use_windowed_read = True
    print(
        f"Will extract {DEFAULT_WINDOW_SIZE}×{DEFAULT_WINDOW_SIZE} pixel window from center"
    )

## Access Green and NIR Bands

Band access is collection-specific:

- **`sentinel2_ard`** — use the multi-band `cog` asset and read Green/NIR by band index (B03/B08).
- **`sentinel-2-c1-l2a`** — use the per-band `green` and `nir` assets directly.

In [ ]:
def _asset_extra(asset):
    if isinstance(asset, dict):
        return asset
    return getattr(asset, "extra_fields", None) or getattr(asset, "extra", None) or {}


def _asset_band_entries(asset):
    extra = _asset_extra(asset)
    return extra.get("eo:bands") or extra.get("bands") or []


def get_scale_offset(asset):
    """Read scale/offset from single-band asset metadata."""
    extra = _asset_extra(asset)
    bands = extra.get("bands", []) or extra.get("raster:bands", [])
    if not bands:
        return 1.0, 0.0
    first = bands[0]
    return float(first.get("scale", 1.0)), float(first.get("offset", 0.0))


def _band_indices_from_eo_bands(asset):
    eo_bands = _asset_band_entries(asset)
    if not eo_bands:
        return None, None

    green_idx = None
    nir_idx = None
    for i, b in enumerate(eo_bands, start=1):
        common_name = str(b.get("common_name") or b.get("eo:common_name") or "").lower()
        name = str(b.get("name") or b.get("eo:name") or "").lower()

        if green_idx is None and (common_name == "green" or name == "b03"):
            green_idx = i
        if nir_idx is None and (common_name == "nir" or name == "b08"):
            nir_idx = i

    return green_idx, nir_idx


def resolve_ndwi_source(item, stac_collection_name):
    collection = str(stac_collection_name or "").strip().lower()

    if collection == "sentinel2_ard":
        if "cog" not in item.assets:
            raise ValueError("Expected 'cog' asset for sentinel2_ard.")

        cog_asset = item.assets["cog"]
        green_idx, nir_idx = _band_indices_from_eo_bands(cog_asset)

        # Fallback to Sentinel-2 defaults if eo:bands is missing.
        if green_idx is None:
            green_idx = 3
        if nir_idx is None:
            nir_idx = 8

        return {
            "mode": "multiband",
            "asset": cog_asset,
            "green_index": green_idx,
            "nir_index": nir_idx,
        }

    if collection == "sentinel-2-c1-l2a":
        if "green" not in item.assets or "nir" not in item.assets:
            raise ValueError("Expected 'green' and 'nir' assets for sentinel-2-c1-l2a.")

        green_asset = item.assets["green"]
        nir_asset = item.assets["nir"]

        return {
            "mode": "separate_assets",
            "green_asset": green_asset,
            "nir_asset": nir_asset,
            "green_scale_offset": get_scale_offset(green_asset),
            "nir_scale_offset": get_scale_offset(nir_asset),
        }

    raise ValueError(
        "Unsupported collection. Expected 'sentinel2_ard' or 'sentinel-2-c1-l2a'."
    )


try:
    source_spec = resolve_ndwi_source(item, stac_collection_name)

    green_scale, green_offset = 1.0, 0.0
    nir_scale, nir_offset = 1.0, 0.0

    if source_spec["mode"] == "separate_assets":
        green_scale, green_offset = source_spec["green_scale_offset"]
        nir_scale, nir_offset = source_spec["nir_scale_offset"]
        print(
            f"Source: separate assets\nGreen: {source_spec['green_asset'].href}\nNIR: {source_spec['nir_asset'].href}"
        )
        if (
            green_scale != 1.0
            or green_offset != 0.0
            or nir_scale != 1.0
            or nir_offset != 0.0
        ):
            print(
                f"Scale/offset: green={green_scale, green_offset}, nir={nir_scale, nir_offset}"
            )
    else:
        print(
            "Source: multiband asset"
            f"\nCOG: {source_spec['asset'].href}"
            f"\nBands: green={source_spec['green_index']}, nir={source_spec['nir_index']}"
        )

except Exception as e:
    print(f"Error accessing bands: {e}")
    print("Available assets:", list(item.assets.keys()))
    raise


## Read Band Data

Read the green and NIR layers from either the `sentinel2_ard` multi-band COG (by band index) or the `sentinel-2-c1-l2a` separate `green`/`nir` assets. Data are clipped to the AOI or to the central 1000×1000 window if no AOI was set. For separate assets, scale and offset from STAC metadata are applied.

In [ ]:
def _resolve_window(src):
    if not use_windowed_read:
        return None

    if src.height < DEFAULT_WINDOW_SIZE or src.width < DEFAULT_WINDOW_SIZE:
        print(
            f"Image is smaller than {DEFAULT_WINDOW_SIZE}x{DEFAULT_WINDOW_SIZE}, using full image"
        )
        return None

    row_off = (src.height - DEFAULT_WINDOW_SIZE) // 2
    col_off = (src.width - DEFAULT_WINDOW_SIZE) // 2
    print(
        f"Extracting {DEFAULT_WINDOW_SIZE}x{DEFAULT_WINDOW_SIZE} pixel window from center"
    )
    return Window(col_off, row_off, DEFAULT_WINDOW_SIZE, DEFAULT_WINDOW_SIZE)


def _read_single_band(src, index):
    if aoi_geometry is not None:
        aoi_in_raster_crs = transform_geom("EPSG:4326", src.crs, aoi_geometry)
        data, transform = mask(src, [aoi_in_raster_crs], crop=True, indexes=[index])
        data = data[0]
        profile = src.profile.copy()
        profile.update(
            {
                "height": data.shape[0],
                "width": data.shape[1],
                "transform": transform,
                "count": 1,
            }
        )
        return data, profile

    clip_window = _resolve_window(src)
    data = src.read(index, window=clip_window)
    profile = src.profile.copy()
    profile.update({"count": 1})
    if clip_window:
        profile.update(
            {
                "height": clip_window.height,
                "width": clip_window.width,
                "transform": rasterio.windows.transform(clip_window, src.transform),
            }
        )
    return data, profile


def _read_multiband_pair(href, green_index, nir_index):
    configure_gdal_auth_for_href(href)
    with rasterio.open(href) as src:
        if src.count < max(green_index, nir_index):
            raise ValueError(
                f"COG has {src.count} bands; need at least band {max(green_index, nir_index)}."
            )
        green_data, green_profile = _read_single_band(src, green_index)
        nir_data, _ = _read_single_band(src, nir_index)
        return green_data, nir_data, green_profile, src.crs


def _read_asset_band(asset, index=1):
    configure_gdal_auth_for_href(asset.href)
    with rasterio.open(asset.href) as src:
        data, profile = _read_single_band(src, index)
        return data, profile, src.crs


try:
    # Ensure AOI variables are initialized (in case AOI cell was not executed)
    try:
        _ = aoi_geometry
    except NameError:
        aoi_geometry = None
        clip_window = None
        use_windowed_read = False

    if source_spec["mode"] == "multiband":
        green_data, nir_data, green_profile, green_crs = _read_multiband_pair(
            source_spec["asset"].href,
            source_spec["green_index"],
            source_spec["nir_index"],
        )
    else:
        green_data, green_profile, green_crs = _read_asset_band(
            source_spec["green_asset"], 1
        )
        nir_data, _, _ = _read_asset_band(source_spec["nir_asset"], 1)

    print(
        f"Read complete: green={green_data.shape} {green_data.dtype}, "
        f"nir={nir_data.shape} {nir_data.dtype}, crs={green_crs}"
    )

    if green_data.shape != nir_data.shape:
        raise ValueError(
            f"Band shapes do not match: Green {green_data.shape} vs NIR {nir_data.shape}"
        )

    if source_spec["mode"] == "multiband":
        green_data = green_data.astype(np.float32)
        nir_data = nir_data.astype(np.float32)
    else:
        green_data = green_data.astype(np.float32) * green_scale + green_offset
        nir_data = nir_data.astype(np.float32) * nir_scale + nir_offset

    print(f"\nBand data loaded successfully! Processing {green_data.size:,} pixels")

except Exception as e:
    print(f"Error reading band data: {e}")
    raise

## Calculate NDWI

$$NDWI = \frac{Green - NIR}{Green + NIR}$$

Valid pixels (where the denominator is non-zero) are computed; invalid pixels are set to NaN. Output values are clamped to the range **[−1, 1]**.

In [ ]:
denominator = green_data + nir_data
valid_mask = denominator != 0
ndwi = np.full_like(green_data, np.nan, dtype=np.float32)
ndwi[valid_mask] = (green_data[valid_mask] - nir_data[valid_mask]) / denominator[
    valid_mask
]
ndwi = np.clip(ndwi, -1.0, 1.0)

print("NDWI calculation complete!")
print(
    f"NDWI min: {np.nanmin(ndwi):.4f}, max: {np.nanmax(ndwi):.4f}, mean: {np.nanmean(ndwi):.4f}"
)
print(f"Valid pixels: {np.sum(~np.isnan(ndwi)):,} of {ndwi.size:,}")

## Visualise Results

Green and NIR bands are displayed with a percentile-based stretch (98th percentile) for contrast. The NDWI panel uses a colour scale from dense vegetation (brown) to water (blue).

In [ ]:
# Create a custom colormap for NDWI visualization
# Colors: dense vegetation (brown) -> sparse vegetation (yellow) -> bare soil (gray) -> water (blue)
colors = ["#8B4513", "#D2691E", "#CCCCCC", "#87CEEB", "#4169E1", "#000080"]
n_bins = 256
cmap = LinearSegmentedColormap.from_list("ndwi", colors, N=n_bins)

# Create figure with subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

vmax_green = (
    np.percentile(green_data[~np.isnan(green_data) & (green_data > 0)], 98)
    if np.any(green_data > 0)
    else np.nanmax(green_data)
)
vmax_nir = (
    np.percentile(nir_data[~np.isnan(nir_data) & (nir_data > 0)], 98)
    if np.any(nir_data > 0)
    else np.nanmax(nir_data)
)
axes[0].imshow(green_data, cmap="Greens", vmin=0, vmax=vmax_green)
axes[0].set_title("Green Band", fontsize=14, fontweight="bold")
axes[0].axis("off")
plt.colorbar(
    axes[0].images[0], ax=axes[0], fraction=0.046, pad=0.04, label="Reflectance"
)

axes[1].imshow(nir_data, cmap="YlGn", vmin=0, vmax=vmax_nir)
axes[1].set_title("NIR Band", fontsize=14, fontweight="bold")
axes[1].axis("off")
plt.colorbar(
    axes[1].images[0], ax=axes[1], fraction=0.046, pad=0.04, label="Reflectance"
)

im3 = axes[2].imshow(ndwi, cmap=cmap, vmin=-1, vmax=1)
axes[2].set_title("NDWI", fontsize=14, fontweight="bold")
axes[2].axis("off")
cbar = plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04, label="NDWI")

# Add colorbar labels
cbar.set_ticks([-1, -0.5, 0, 0.3, 0.6, 1])
cbar.set_ticklabels(
    [
        "Dense Veg",
        "Sparse Veg",
        "Bare Soil",
        "Moist Soil",
        "Shallow Water",
        "Deep Water",
    ]
)

plt.suptitle(f"NDWI — {item.id}", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Visualisation complete.")

## Summary

- **Supported data**: `sentinel2_ard` (multi-band `cog`) and `sentinel-2-c1-l2a` (`green`/`nir` assets).
- **Auth**: Use a token in `.env` only for authenticated catalogues; public data work without it.
- **Steps**: Load STAC item → resolve green/NIR by collection → read data (with scale/offset for separate assets) → compute NDWI → visualise.
- **Next**: Change `stac_item_url` and `stac_collection_name` for other scenes; optionally set `aoi_param` to a GeoJSON geometry.